# Crew Planning

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) CrewAI roadmap — see **Crew Planning** on the CrewAI path.

You will enable `planning=True` on a crew, run `kickoff`, and observe how planning augments execution (verbose logs show planner-related activity).

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Crew with `planning=True`

Before tasks run, CrewAI’s internal **AgentPlanner** builds step-by-step plans from your crew configuration and **injects** them into each task’s description. With **`verbose=True`**, you can follow orchestration and see how agents receive structured guidance compared to starting from raw descriptions alone.

Run `kickoff` and inspect the printed trace and final `CrewOutput`.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Gather concise, accurate facts on the topic",
    backstory="You summarize sources clearly and avoid speculation.",
    verbose=True,
)

analyst = Agent(
    role="Analyst",
    goal="Extract themes and implications from research",
    backstory="You connect facts to clear takeaways.",
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Produce a short reader-facing summary",
    backstory="You write tight prose for a general audience.",
    verbose=True,
)

research_task = Task(
    description="Research the topic: {topic}. List 3–5 bullet facts.",
    expected_output="A short bullet list of facts.",
    agent=researcher,
)

analysis_task = Task(
    description="From the research, identify 2–3 themes and one implication each.",
    expected_output="A brief analysis with themes and implications.",
    agent=analyst,
    context=[research_task],
)

writing_task = Task(
    description="Write two sentences summarizing the analysis for a busy reader.",
    expected_output="Exactly two sentences.",
    agent=writer,
    context=[analysis_task],
)

crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    planning=True,
    planning_llm="gpt-4o",
    verbose=True,
)

result = crew.kickoff(inputs={"topic": "renewable energy trends"})
print(result)

## Key takeaways

- **`planning=True`** runs an internal **AgentPlanner** before tasks execute.
- The planner’s output is **injected into task descriptions** so agents get step-by-step guidance.
- **`planning_llm`** sets the model for that planning call (default is **`gpt-4o-mini`**).
- Expect a small **extra latency and token cost** for the planning step.
- Best for **complex, multi-step** crews when **consistency** and **structured execution** matter.